# Support Instructor Demo — Retrieval on a Recipe Corpus

**Domain:** Cooking recipes (200 documents from `m3hrdadfi/recipe_nlg_lite`).

**Why this domain:** Same procedural pattern as the Module 8A lab, but a different surface (ingredients / techniques) and a different data shape (short, structured rows). Learners see the *mechanics* without being able to copy the demo as their submission.

**Demo timing:** ~30 minutes, Day 2 morning. Lab releases at noon.

**Flow (~30 min):**
1. **Frame** (3 min) — Show the corpus.
2. **Schema + ingest** (5 min) — Create the `Post` class, index 200 docs.
3. **BM25** (7 min) — Rare-token queries; lexical wins.
4. **Dense** (7 min) — Paraphrastic queries; embeddings win.
5. **Hybrid + α sweep** (5 min) — One mixed query at α ∈ {0.0, 0.5, 1.0}.
6. **Pivot to lab** (3 min) — Same machinery, new domain — go.

## 0. Setup — imports and Weaviate client

Weaviate is expected to be running on port **8090** (not 8080) so it does not collide with the learner's lab container. The gRPC port is also remapped (`50061`→`50051`) for the same reason. Start it with:

```bash
docker run -d --name weaviate-si-demo \
  -p 8090:8080 \
  -p 50061:50051 \
  -e DEFAULT_VECTORIZER_MODULE=none \
  -e ENABLE_MODULES= \
  -e PERSISTENCE_DATA_PATH=/var/lib/weaviate \
  semitechnologies/weaviate:1.27.0
```

In [1]:
%%bash
# -d: run detached | --name: container name for easy reference
# -p 8090:8080: remap HTTP port to avoid collision with the lab container on 8080
# -p 50061:50051: remap gRPC port away from the lab's 50051
# DEFAULT_VECTORIZER_MODULE=none: we supply vectors manually, no built-in vectorizer needed
# ENABLE_MODULES=: no extra Weaviate modules required
# PERSISTENCE_DATA_PATH: where Weaviate writes its data inside the container
docker run -d --name weaviate-si-demo \
  -p 8090:8080 \
  -p 50061:50051 \
  -e DEFAULT_VECTORIZER_MODULE=none \
  -e ENABLE_MODULES= \
  -e PERSISTENCE_DATA_PATH=/var/lib/weaviate \
  semitechnologies/weaviate:1.27.0

f1804ff0503d2212268f47b9afaf6f082b6eaea619309c1e5a331523524519bc


In [2]:
import json                                                          # standard library JSON parser
from pathlib import Path                                            # cross-platform filesystem paths

import weaviate                                                     # Weaviate Python client v4
from weaviate.classes.config import Configure, DataType, Property  # schema builder helpers
from weaviate.classes.query import MetadataQuery                    # metadata selectors for scores/distances
from sentence_transformers import SentenceTransformer              # bi-encoder embedding model
from datasets import load_dataset                                   # HuggingFace datasets loader

WEAVIATE_HTTP_PORT = 8090                                          # non-default; avoids collision with lab's 8080
WEAVIATE_GRPC_PORT = 50061                                         # non-default; avoids collision with lab's 50051
CLASS_NAME = "Post"                                                # Weaviate collection name (matches lab schema)
CORPUS_PATH = Path("demo_corpus.jsonl")                            # path to the local recipe JSONL corpus
EMBEDDER_NAME = "sentence-transformers/all-MiniLM-L6-v2"          # compact 384-dim bi-encoder model

client = weaviate.connect_to_local(port=WEAVIATE_HTTP_PORT, grpc_port=WEAVIATE_GRPC_PORT)  # connect to local Weaviate instance
assert client.is_ready(), "Weaviate is not reachable on port 8090. Check the docker container."  # halt early if Weaviate is down
print("Weaviate ready on port", WEAVIATE_HTTP_PORT)                # confirm successful connection

/home/brandon/.local/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/usr/lib/python3/dist-packages/pytz/__init__.py:31: SyntaxWarning: invalid escape sequence '\s'
  match = re.match("^#\s*version\s*([0-9a-z]*)\s*$", line)
/usr/lib/python3/dist-packages/pytz/tzinfo.py:27: DeprecationWarning: datetime.datetime.utcfromtimestamp() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.fromtimestamp(timestamp, datetime.UTC).
  _epoch = datetime.utcfromtimestamp(0)
<frozen importlib._bootstrap>:488: DeprecationWarning: builtin type SwigPyPacked has no __module__ attribute
<frozen importlib._bootstrap>:488: DeprecationWarning: builtin type SwigPyObject has no __module__ attribute


Weaviate ready on port 8090


## 1. Frame — show the corpus (0:00 – 0:03)

> *"Same schema as the lab corpus — different domain. Watch what changes."*

Pull 200 recipes and reshape them into the lab's `Post` schema so `create_schema`, `index_corpus`, and the three `*_search` functions run unchanged.

In [3]:
if not CORPUS_PATH.exists():                                        # skip download if corpus is already on disk
    # The dataset has a Python loading script which datasets>=3.0 rejects;
    # load the Hub's auto-converted parquet snapshot instead.
    RECIPES_PARQUET = "https://huggingface.co/datasets/m3hrdadfi/recipe_nlg_lite/resolve/refs%2Fconvert%2Fparquet/1.0.0/recipe_nlg_lite-train.parquet"  # pre-converted parquet URL
    ds = load_dataset("parquet", data_files=RECIPES_PARQUET, split="train")  # load all rows as a HF Dataset
    recipes = ds.select(range(200))                                 # keep only first 200 recipes for the demo

    with CORPUS_PATH.open("w") as f:                               # open JSONL file for writing
        for i, r in enumerate(recipes):                            # iterate with numeric index for stable IDs
            doc = {
                "id": f"recipes:{i}",                              # stable unique ID per document
                "subset": "recipes",                               # tag for corpus provenance
                "title": r["name"],                                # recipe name → title field
                "question_text": r["ingredients"],                 # ingredients list → mirrors lab schema field
                "answer_text": r["steps"],                         # cooking steps → mirrors lab schema field
                "text": f"{r['name']}\n\n{r['ingredients']}\n\nSteps: {r['steps']}",  # full concatenated text for embedding and BM25
                "license_version": "see dataset card",             # provenance note
                "source_url": "https://huggingface.co/datasets/m3hrdadfi/recipe_nlg_lite",  # original dataset URL
            }
            f.write(json.dumps(doc) + "\n")                        # write one JSON object per line (JSONL format)
    print(f"Wrote {CORPUS_PATH} with 200 recipes.")                # confirm write success
else:
    print(f"{CORPUS_PATH} already exists — reusing.")              # skip and reuse the cached file

# Eyeball the first 5 lines so the room sees the shape.
with CORPUS_PATH.open() as f:                                      # open corpus for reading
    for _ in range(5):                                             # print exactly 5 sample rows
        row = json.loads(f.readline())                             # parse one JSONL line at a time
        print(f"- {row['id']}: {row['title']}")                    # show id and title for a quick sanity check

demo_corpus.jsonl already exists — reusing.
- recipes:0: pork chop noodle soup
- recipes:1: 5 ingredient almond cake with fresh berries
- recipes:2: shrimp cakes
- recipes:3: chili roasted okra
- recipes:4: slow cooker chicken chili


## 2. Schema + ingest (0:03 – 0:08)

> *"Same code path as your lab. The schema doesn't care about the domain."*

**`create_schema`** — define a `Post` class with no vectorizer (we'll supply vectors manually so the demo runs offline).

**`index_corpus`** — encode each row with `all-MiniLM-L6-v2` and batch-insert.

In [4]:
def create_schema():
    if client.collections.exists(CLASS_NAME):            # check whether the collection already exists
        client.collections.delete(CLASS_NAME)            # drop it so re-runs start from a clean slate
    client.collections.create(
        name=CLASS_NAME,                                 # name the collection "Post" (matches lab schema)
        vectorizer_config=Configure.Vectorizer.none(),   # no built-in vectorizer; we supply vectors manually
        properties=[
            Property(name="doc_id",          data_type=DataType.TEXT),  # original corpus ID string
            Property(name="subset",          data_type=DataType.TEXT),  # corpus source tag
            Property(name="title",           data_type=DataType.TEXT),  # recipe name
            Property(name="question_text",   data_type=DataType.TEXT),  # ingredients list
            Property(name="answer_text",     data_type=DataType.TEXT),  # cooking steps
            Property(name="text",            data_type=DataType.TEXT),  # full concatenated text (BM25 indexes this)
            Property(name="license_version", data_type=DataType.TEXT),  # dataset license info
            Property(name="source_url",      data_type=DataType.TEXT),  # link back to HuggingFace
        ],
    )
    print(f"Created class {CLASS_NAME}.")                # confirm the schema was created successfully

create_schema()                                          # execute immediately to set up the collection

/home/brandon/.local/lib/python3.13/site-packages/weaviate/warnings.py:196: DeprecationWarning: Dep024: You are using the `vectorizer_config` argument in `collection.config.create()`, which is deprecated.
            Use the `vector_config` argument instead.
            
  warnings.warn(


Created class Post.


In [5]:
embedder = SentenceTransformer(EMBEDDER_NAME)  # load all-MiniLM-L6-v2 (~22 MB, 384-dim output)
print("Loaded embedder:", EMBEDDER_NAME)       # confirm the model loaded without error

Loading weights: 100%|██████████| 103/103 [00:01<00:00, 75.13it/s]


Loaded embedder: sentence-transformers/all-MiniLM-L6-v2


In [6]:
def index_corpus(path: Path):
    rows = [json.loads(line) for line in path.open()]    # load all JSONL rows into memory at once
    texts = [r["text"] for r in rows]                   # extract the full-text field for embedding
    vectors = embedder.encode(                           # encode all 200 texts in batches
        texts,
        batch_size=32,                                   # 32 docs per inference call; balances speed and memory
        show_progress_bar=True,                          # display a tqdm bar during encoding
        normalize_embeddings=True,                       # L2-normalize so cosine similarity = dot product
    )

    posts = client.collections.get(CLASS_NAME)          # get a handle to the "Post" collection
    with posts.batch.dynamic() as batch:                # dynamic batching auto-tunes the batch size at runtime
        for r, vec in zip(rows, vectors):               # pair each document dict with its embedding
            batch.add_object(
                properties={
                    "doc_id":          r["id"],          # original corpus ID
                    "subset":          r["subset"],      # corpus provenance tag
                    "title":           r["title"],       # recipe name
                    "question_text":   r["question_text"],  # ingredients list
                    "answer_text":     r["answer_text"], # cooking steps
                    "text":            r["text"],        # full text (Weaviate BM25 indexes this field)
                    "license_version": r["license_version"],  # license string
                    "source_url":      r["source_url"],  # source dataset URL
                },
                vector=vec.tolist(),                     # attach dense vector (numpy array → Python list)
            )
    print(f"Indexed {len(rows)} documents.")             # confirm all documents were inserted

index_corpus(CORPUS_PATH)                               # kick off ingest for the 200-recipe corpus

/tmp/ipykernel_110276/2796189801.py:2: ResourceWarning: unclosed file <_io.TextIOWrapper name='demo_corpus.jsonl' mode='r' encoding='UTF-8'>
  rows = [json.loads(line) for line in path.open()]    # load all JSONL rows into memory at once
Batches: 100%|██████████| 7/7 [00:38<00:00,  5.45s/it]


Indexed 200 documents.


In [7]:
# Confirm the aggregate count — should be 200.
posts = client.collections.get(CLASS_NAME)               # get a handle to the Post collection
count = posts.aggregate.over_all(total_count=True).total_count  # ask Weaviate for the total document count
print(f"Aggregate count: {count}")                       # display the count for the room to verify
assert count == 200, "Expected 200 documents — did ingest fail?"  # halt immediately if any documents failed to insert

Aggregate count: 200


## 3. Retrieval functions

Three thin wrappers around Weaviate's query API — one per retrieval mode. Each returns the top-k rows with their title and relevance score.

In [8]:
def _format(results):
    rows = []
    for i, obj in enumerate(results.objects, start=1):   # rank starting from 1 (not 0)
        # BM25 results carry .score; dense (near_vector) results carry .distance — handle both
        score = obj.metadata.score if obj.metadata.score is not None else obj.metadata.distance
        rows.append({
            "rank":  i,
            "title": obj.properties["title"],
            "score": round(float(score), 4),             # round to 4 dp for clean display
        })
    return rows

def bm25_search(query: str, k: int = 5):
    posts = client.collections.get(CLASS_NAME)           # get collection handle each call (client is stateless)
    res = posts.query.bm25(                              # lexical BM25 search
        query=query,
        limit=k,                                         # return top-k results
        return_metadata=MetadataQuery(score=True),       # include BM25 score in each result object
    )
    return _format(res)                                  # normalise to a plain ranked list

def dense_search(query: str, k: int = 5):
    posts = client.collections.get(CLASS_NAME)           # get collection handle
    qvec = embedder.encode(query, normalize_embeddings=True).tolist()  # encode query; L2-normalize to match stored vectors
    res = posts.query.near_vector(                       # approximate nearest-neighbour search
        near_vector=qvec,
        limit=k,                                         # return top-k results
        return_metadata=MetadataQuery(distance=True),    # include cosine distance in each result object
    )
    return _format(res)                                  # normalise to a plain ranked list

def hybrid_search(query: str, alpha: float = 0.5, k: int = 5):
    posts = client.collections.get(CLASS_NAME)           # get collection handle
    qvec = embedder.encode(query, normalize_embeddings=True).tolist()  # encode query for the dense side
    res = posts.query.hybrid(
        query=query,                                     # raw text for the BM25 side
        vector=qvec,                                     # embedding for the dense side
        alpha=alpha,                                     # 0.0 = pure BM25 | 1.0 = pure dense | 0.5 = balanced
        limit=k,                                         # return top-k fused results
        return_metadata=MetadataQuery(score=True),       # include the fused hybrid score
    )
    return _format(res)                                  # normalise to a plain ranked list

def show(rows, header):
    print(f"\n=== {header} ===")                         # section divider with the query label
    for r in rows:
        print(f"  {r['rank']:>2}. [{r['score']:>7}]  {r['title']}")  # right-align rank and score for easy scanning

## 4. BM25 demo (0:08 – 0:15)

> *"Rare tokens win on lexical match — same pattern you'll see with `EACCES` in the lab."*

**Bucket 1 — BM25-favorable.** Each query contains a rare ingredient that appears verbatim in only a handful of recipes. Lexical match dominates.

In [9]:
bm25_queries = [
    "harissa paste",       # rare ingredient; appears verbatim in very few recipes — strong lexical signal
    "panko breadcrumbs",   # another rare token; BM25 should surface relevant recipes easily
    # spares (pre-tested Sunday):
    # "espelette pepper", "preserved lemon"
]

for q in bm25_queries:
    show(bm25_search(q), f"BM25 — {q!r}")                       # lexical results — expected to be on-topic
    show(dense_search(q), f"Dense — {q!r}  (for contrast)")     # dense results — expected to drift to unrelated recipes


=== BM25 — 'harissa paste' ===
   1. [ 1.4948]  tuscan roast leg of pork
   2. [ 1.3628]  thai chicken coconut soup tom kha gai recipe
   3. [ 1.3531]  red thai coconut curry shrimp
   4. [  1.302]  garlic herb roast pork
   5. [  1.038]  thai fish cakes with spicy sauce todman bla

=== Dense — 'harissa paste'  (for contrast) ===
   1. [  0.699]  adasi persian lentil stew
   2. [ 0.7039]  salad bong cai xanh va hat dieu
   3. [ 0.7334]  orange fennel marmalade
   4. [ 0.7621]  tortino d'azzima matzo pie
   5. [ 0.7695]  skinny garlic mashed potatoes

=== BM25 — 'panko breadcrumbs' ===
   1. [  2.863]  mustard herb crusted pork loin roast
   2. [ 2.7672]  buffalo baked chicken tenders
   3. [ 1.8399]  shrimp cakes
   4. [ 1.6606]  stuffed mashed potato tots
   5. [ 1.2648]  brussels sprouts gratin with bacon breadcrumbs

=== Dense — 'panko breadcrumbs'  (for contrast) ===
   1. [ 0.6228]  classic meat loaf
   2. [ 0.6706]  mapo eggplant mabo nasu
   3. [ 0.6725]  banana bread
   4. [ 0

## 5. Dense demo (0:15 – 0:22)

> *"No shared words but the right recipes still surface. This is the embedding doing the work."*

**Bucket 2 — dense-favorable.** Each query expresses *intent* with vocabulary the corpus does not literally use.

In [10]:
dense_queries = [
    "quick weeknight dinner with what's in the fridge",  # intent-only; no literal corpus vocabulary
    "comfort food for a cold day",                        # paraphrastic; relies entirely on semantic similarity
    # spare:
    # "something light and refreshing for summer"
]

for q in dense_queries:
    show(bm25_search(q), f"BM25 — {q!r}  (for contrast)")  # BM25 matches on stop-words; results will be weak/off-topic
    show(dense_search(q), f"Dense — {q!r}")                 # dense expected to surface semantically relevant recipes


=== BM25 — "quick weeknight dinner with what's in the fridge"  (for contrast) ===
   1. [ 2.6435]  cheeseburger dinner
   2. [ 2.1613]  quick string cheese spaghetti and meatballs
   3. [ 2.0103]  kimchi fried rice with sesame egg
   4. [ 1.9853]  spicy quick fix chili
   5. [ 1.9768]  how to make a vegan egg substitute

=== Dense — "quick weeknight dinner with what's in the fridge" ===
   1. [ 0.5866]  just for mom, salad
   2. [ 0.5897]  balsamic chicken, tangerines, dried plums and baby greens
   3. [ 0.5923]  knock your stilettos off shooters
   4. [  0.609]  sweet and spicy chip dip
   5. [ 0.6173]  ciabatta cold cut sandwich with coleslaw

=== BM25 — 'comfort food for a cold day'  (for contrast) ===
   1. [ 2.4482]  ham pinwheels
   2. [  1.294]  butternut squash "rice" risotto with shrimp and peas
   3. [ 1.1813]  ciabatta cold cut sandwich with coleslaw
   4. [ 1.0904]  sea glass rock candy
   5. [ 1.0795]  easy no cook salsa recipe

=== Dense — 'comfort food for a cold day' =

## 6. Hybrid + alpha sweep (0:22 – 0:27)

> *"Hybrid isn't magic — it's a knob. Alpha is empirical; the lab asks you to find a good value for the Stack Exchange corpus."*

α controls the BM25-vs-dense mix:
- **α = 0.0** → pure BM25
- **α = 1.0** → pure dense
- **α = 0.5** → balanced

**Bucket 3 — mixed signal.** Has both a rare-ish token (`pasta`, `chicken`) and an intent component (`easy 30-minute`).

In [11]:
hybrid_query = "easy 30-minute pasta with chicken"  # mixed: rare tokens ("pasta","chicken") + intent phrasing ("easy","30-minute")

for alpha in (0.0, 0.5, 1.0):                       # sweep three representative alpha values
    show(                                            # print the ranked results for this alpha
        hybrid_search(hybrid_query, alpha=alpha),
        f"Hybrid α={alpha} — {hybrid_query!r}",      # label shows alpha so learners can compare the three outputs
    )


=== Hybrid α=0.0 — 'easy 30-minute pasta with chicken' ===
   1. [    1.0]  easy and basic chicken child friendly noodle soup recipe
   2. [ 0.6047]  quick string cheese spaghetti and meatballs
   3. [ 0.5839]  garlic chicken and shells
   4. [ 0.5751]  mushroom pepper pasta with feta goat cheese sauce
   5. [ 0.5522]  easy chicken and rice casserole

=== Hybrid α=0.5 — 'easy 30-minute pasta with chicken' ===
   1. [ 0.9598]  easy and basic chicken child friendly noodle soup recipe
   2. [ 0.7668]  pasta with cauliflower
   3. [  0.766]  garlic chicken and shells
   4. [ 0.6999]  quick string cheese spaghetti and meatballs
   5. [ 0.6481]  chicken sausage and peppers macaroni casserole

=== Hybrid α=1.0 — 'easy 30-minute pasta with chicken' ===
   1. [    1.0]  pasta with cauliflower
   2. [ 0.9481]  garlic chicken and shells
   3. [ 0.9195]  easy and basic chicken child friendly noodle soup recipe
   4. [ 0.9063]  skillet cheesy chicken and veggie "rice"
   5. [ 0.8936]  spaghetti an

## 7. Pivot to lab (0:27 – 0:30)

> *"Different domain, same machinery. Today you do this for 1,200 Stack Exchange posts and 60 labeled queries — and write a 300–500 word brief on which retriever wins which kind of query, and why."*

Drop the lab guide URL in chat. End of demo.

In [12]:
client.close()                                  # release the gRPC and HTTP connection pool cleanly
print("Closed Weaviate client. Demo complete.")  # confirm shutdown for the room

Closed Weaviate client. Demo complete.


In [13]:
# Tear down the Weaviate container so cell-1's `docker run` succeeds next time
# (otherwise it errors with "name already in use").
!docker stop weaviate-si-demo  # send SIGTERM; Docker waits up to 10 s for graceful shutdown
!docker rm weaviate-si-demo    # remove the stopped container so the name can be reused next run

/usr/lib/python3.13/pty.py:95: DeprecationWarning: This process (pid=110276) is multi-threaded, use of forkpty() may lead to deadlocks in the child.
  pid, fd = os.forkpty()


weaviate-si-demo
weaviate-si-demo


---

## Appendix A — The Haystack platform

Everything above was built by hand: a Weaviate client, an explicit schema, manual encoding with `sentence-transformers`, and three separate retrieval functions wired to the Weaviate query API. That's useful for *teaching* — learners see each moving part — but it's a lot of glue code for what is, in the end, a fairly standard RAG pipeline.

**[Haystack](https://haystack.deepset.ai)** (deepset, `pip install haystack-ai`) is an open-source framework that compresses the same pipeline into a handful of named components:

| Stage | Hand-rolled version above | Haystack equivalent |
|---|---|---|
| Store documents | Weaviate `Post` class with manual schema | `InMemoryDocumentStore()` |
| Embed text | `SentenceTransformer(...).encode(...)` | `SentenceTransformersDocumentEmbedder(model=...)` + `SentenceTransformersTextEmbedder(model=...)` |
| BM25 retrieve | `posts.query.bm25(query=...)` | `InMemoryBM25Retriever(document_store=...).run(query=...)` |
| Dense retrieve | `posts.query.near_vector(near_vector=...)` | `InMemoryEmbeddingRetriever(document_store=...).run(query_embedding=...)` |
| Hybrid retrieve | `posts.query.hybrid(query=..., vector=..., alpha=...)` | Run both retrievers, fuse with `DocumentJoiner(join_mode="reciprocal_rank_fusion")` |

**What you trade.** Haystack hides the vector store (it ships an in-memory store with a real BM25 implementation — `rank_bm25` under the hood) and replaces Weaviate's continuous `alpha` knob with rank-fusion strategies (`reciprocal_rank_fusion`, `distribution_based_rank_fusion`, plain `concatenate`, `merge`). You give up the alpha sweep but you keep true BM25 — a different trade-off than other low-code frameworks.

**When to reach for it.** Prototyping; teaching the *shape* of a RAG pipeline without drowning in plumbing; building production pipelines where you want declarative `Pipeline` graphs and integration points for evaluators, generators, and routers. When you need precise control over hybrid scoring (continuous alpha, ANN tuning, server-side BM25), go back to the Weaviate path above.

The cells below re-run the same recipe demo through Haystack so you can compare the two surfaces side-by-side.

### A.1 — Install + imports

If Haystack isn't on the kernel yet, run the install cell once. `sentence-transformers` should already be installed from the Weaviate section above.

```bash
pip install haystack-ai
```

In [14]:
!pip install haystack-ai -q  # install Haystack; -q suppresses verbose pip output


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [15]:
from haystack import Document                                          # Haystack's base document class
from haystack.document_stores.in_memory import InMemoryDocumentStore  # in-memory store with built-in BM25 + vector index (no external DB)
from haystack.components.embedders import (
    SentenceTransformersDocumentEmbedder,                             # batch-embeds a list of Document objects
    SentenceTransformersTextEmbedder,                                 # embeds a single query string at inference time
)
from haystack.components.retrievers.in_memory import (
    InMemoryBM25Retriever,                                            # BM25 retriever backed by rank_bm25
    InMemoryEmbeddingRetriever,                                       # ANN retriever using cosine similarity
)
from haystack.components.joiners import DocumentJoiner                # fuses multiple ranked lists (e.g. RRF, concatenate)

EMBEDDER_NAME = "sentence-transformers/all-MiniLM-L6-v2"             # same model as the Weaviate section above for a fair comparison
print("Haystack imported.")                                           # confirm all imports succeeded

Haystack imported.


### A.2 — Ingest corpus into InMemoryDocumentStore

Load the JSONL corpus, embed each document with the same `all-MiniLM-L6-v2` model, and write to an in-memory store. `SentenceTransformersDocumentEmbedder` handles batching automatically.

In [16]:
store = InMemoryDocumentStore()                                       # create in-memory store; builds BM25 index automatically on write

with CORPUS_PATH.open() as f:                                        # open the same JSONL corpus used in the Weaviate section
    raw_docs = [json.loads(line) for line in f]                      # parse all 200 lines into dicts

haystack_docs = [
    Document(                                                        # wrap each row in a Haystack Document
        content=row["text"],                                         # full concatenated text — used for both BM25 and embedding
        meta={"title": row["title"], "doc_id": row["id"]},          # store title and id in metadata for display
    )
    for row in raw_docs
]

doc_embedder = SentenceTransformersDocumentEmbedder(model=EMBEDDER_NAME)  # embedder that operates on Document objects
doc_embedder.warm_up()                                               # load model weights into memory before encoding
embedded = doc_embedder.run(haystack_docs)                           # encode all docs; returns {"documents": [...]} with .embedding set on each
store.write_documents(embedded["documents"])                         # write embedded docs to store (also indexes them for BM25)
print(f"Indexed {store.count_documents()} documents.")              # confirm all 200 docs are stored

The `tokenizer_kwargs` argument was renamed and is now deprecated. Please use `processor_kwargs` instead.
Batches: 100%|██████████| 7/7 [00:22<00:00,  3.21s/it]

Indexed 200 documents.


### A.3 — Retrieval functions

Three thin wrappers — one per mode — mirroring `bm25_search`, `dense_search`, and `hybrid_search` from the Weaviate section. `show_hs` prints results in the same format as `show`.

In [17]:
text_embedder = SentenceTransformersTextEmbedder(model=EMBEDDER_NAME)  # query-side embedder (single string → vector)
text_embedder.warm_up()                                              # load model weights once; reused across all queries

bm25_ret  = InMemoryBM25Retriever(document_store=store)             # BM25 retriever wired to our in-memory store
dense_ret = InMemoryEmbeddingRetriever(document_store=store)        # ANN retriever wired to our in-memory store
joiner    = DocumentJoiner(join_mode="reciprocal_rank_fusion")      # RRF fusion: rank-based, immune to BM25/dense score-scale differences


def hs_bm25(query, k=5):
    return bm25_ret.run(query=query, top_k=k)["documents"]          # lexical retrieval; returns top-k Document objects


def hs_dense(query, k=5):
    qvec = text_embedder.run(text=query)["embedding"]               # encode query string to a vector
    return dense_ret.run(query_embedding=qvec, top_k=k)["documents"]  # ANN retrieval using the query vector


def hs_hybrid(query, k=5):
    bm25_docs  = bm25_ret.run(query=query, top_k=k)["documents"]    # BM25 candidate list
    qvec       = text_embedder.run(text=query)["embedding"]         # encode query for the dense side
    dense_docs = dense_ret.run(query_embedding=qvec, top_k=k)["documents"]  # dense candidate list
    fused      = joiner.run(documents=[bm25_docs, dense_docs])["documents"]  # merge with RRF; scores are now rank-based
    return fused[:k]                                                 # trim to k after fusion (joiner may return more)


def show_hs(docs, header):
    print(f"\n=== {header} ===")                                     # section divider with query label
    for i, doc in enumerate(docs, 1):                               # rank starting from 1
        score = round(float(doc.score), 4) if doc.score is not None else 0.0  # use doc.score; default 0.0 if absent (RRF always sets it)
        print(f"  {i:>2}. [{score:>7}]  {doc.meta.get('title', '(untitled)')}")  # right-aligned rank/score, then recipe title

### A.4 — BM25-favorable queries

Rare tokens win on lexical match — same pattern as the Weaviate demo. Haystack's `InMemoryBM25Retriever` uses `rank_bm25` under the hood.

In [18]:
for q in ["harissa paste", "panko breadcrumbs"]:
    show_hs(hs_bm25(q),  f"BM25  — {q!r}")             # lexical wins on rare tokens; same pattern as the Weaviate demo
    show_hs(hs_dense(q), f"Dense — {q!r}  (contrast)")  # dense drifts on specific ingredient queries


=== BM25  — 'harissa paste' ===
   1. [ 5.3175]  tuscan roast leg of pork
   2. [ 5.1954]  red thai coconut curry shrimp
   3. [ 5.0911]  thai chicken coconut soup tom kha gai recipe
   4. [ 4.9796]  garlic herb roast pork
   5. [ 4.6377]  thai fish cakes with spicy sauce todman bla


Batches: 100%|██████████| 1/1 [00:00<00:00, 43.20it/s]



=== Dense — 'harissa paste'  (contrast) ===
   1. [  0.301]  adasi persian lentil stew
   2. [ 0.2961]  salad bong cai xanh va hat dieu
   3. [ 0.2666]  orange fennel marmalade
   4. [ 0.2379]  tortino d'azzima matzo pie
   5. [ 0.2305]  skinny garlic mashed potatoes

=== BM25  — 'panko breadcrumbs' ===
   1. [10.9437]  mustard herb crusted pork loin roast
   2. [10.3036]  buffalo baked chicken tenders
   3. [ 8.6264]  shrimp cakes
   4. [ 8.3948]  stuffed mashed potato tots
   5. [ 7.3712]  brussels sprouts gratin with bacon breadcrumbs


Batches: 100%|██████████| 1/1 [00:00<00:00, 49.86it/s]


=== Dense — 'panko breadcrumbs'  (contrast) ===
   1. [ 0.3772]  classic meat loaf
   2. [ 0.3294]  mapo eggplant mabo nasu
   3. [ 0.3275]  banana bread
   4. [ 0.3216]  fresh radish tartines with cultured butter and sea salt
   5. [ 0.3096]  ciabatta cold cut sandwich with coleslaw


### A.5 — Dense-favorable queries

Intent-only queries with no literal vocabulary overlap in the corpus — embeddings win.

In [19]:
for q in [
    "quick weeknight dinner with what's in the fridge",
    "comfort food for a cold day",
]:
    show_hs(hs_bm25(q),  f"BM25  — {q!r}  (contrast)")  # BM25 latches onto stop-words; weak relevance
    show_hs(hs_dense(q), f"Dense — {q!r}")               # dense surfaces semantically relevant recipes


=== BM25  — "quick weeknight dinner with what's in the fridge"  (contrast) ===
   1. [18.3085]  how to make a vegan egg substitute
   2. [17.8788]  cheeseburger dinner
   3. [17.0335]  kimchi fried rice with sesame egg
   4. [16.8942]  quick string cheese spaghetti and meatballs
   5. [16.2373]  spicy quick fix chili


Batches: 100%|██████████| 1/1 [00:00<00:00, 37.83it/s]



=== Dense — "quick weeknight dinner with what's in the fridge" ===
   1. [ 0.4134]  just for mom, salad
   2. [ 0.4103]  balsamic chicken, tangerines, dried plums and baby greens
   3. [ 0.4077]  knock your stilettos off shooters
   4. [  0.391]  sweet and spicy chip dip
   5. [ 0.3827]  ciabatta cold cut sandwich with coleslaw

=== BM25  — 'comfort food for a cold day'  (contrast) ===
   1. [11.6281]  ham pinwheels
   2. [ 9.8111]  petits financiers little almond cakes
   3. [ 9.8068]  banana bread
   4. [ 9.5326]  butternut squash "rice" risotto with shrimp and peas
   5. [ 9.4997]  rotisserie turkey breast tenderloin with pesto


Batches: 100%|██████████| 1/1 [00:00<00:00, 42.45it/s]


=== Dense — 'comfort food for a cold day' ===
   1. [ 0.3454]  spicy tortilla soup
   2. [ 0.3285]  chili a la jimmy fallon
   3. [ 0.3273]  ciabatta cold cut sandwich with coleslaw
   4. [ 0.3226]  sandra's mexican israeli salad
   5. [ 0.3177]  skinny er shamrock shake


### A.6 — Hybrid (Reciprocal Rank Fusion)

Haystack's `DocumentJoiner` with `join_mode="reciprocal_rank_fusion"` replaces Weaviate's continuous `alpha` knob. Both result lists are interleaved using RRF scores — rank-based, so immune to BM25/dense score-scale differences.

In [20]:
mixed_query = "easy 30-minute pasta with chicken"                           # mixed signal: rare tokens + intent phrasing
show_hs(hs_bm25(mixed_query),   f"BM25         — {mixed_query!r}")          # pure lexical ranking baseline
show_hs(hs_dense(mixed_query),  f"Dense        — {mixed_query!r}")          # pure semantic ranking baseline
show_hs(hs_hybrid(mixed_query), f"Hybrid (RRF) — {mixed_query!r}")          # RRF fusion combines both signals; compare rank shifts vs. the two baselines


=== BM25         — 'easy 30-minute pasta with chicken' ===
   1. [14.3049]  easy and basic chicken child friendly noodle soup recipe
   2. [ 11.545]  mushroom pepper pasta with feta goat cheese sauce
   3. [11.5226]  garlic chicken and shells
   4. [11.3934]  j . alexander's mac and cheese
   5. [11.1447]  pork chop noodle soup


Batches: 100%|██████████| 1/1 [00:00<00:00, 32.33it/s]



=== Dense        — 'easy 30-minute pasta with chicken' ===
   1. [ 0.5753]  pasta with cauliflower
   2. [ 0.5621]  garlic chicken and shells
   3. [ 0.5549]  easy and basic chicken child friendly noodle soup recipe
   4. [ 0.5516]  skillet cheesy chicken and veggie "rice"
   5. [ 0.5484]  spaghetti and quinoa meatballs


Batches: 100%|██████████| 1/1 [00:00<00:00, 44.23it/s]


=== Hybrid (RRF) — 'easy 30-minute pasta with chicken' ===
   1. [ 0.9841]  easy and basic chicken child friendly noodle soup recipe
   2. [ 0.9761]  garlic chicken and shells
   3. [    0.5]  pasta with cauliflower
   4. [ 0.4919]  mushroom pepper pasta with feta goat cheese sauce
   5. [ 0.4766]  j . alexander's mac and cheese


### A.7 — Takeaways

The same patterns hold as in the Weaviate section: rare-token queries benefit from BM25's keyword side; intent-only queries favor the dense retriever; the mixed query benefits from combining both signals.

The key API difference: Haystack offers discrete fusion strategies (`reciprocal_rank_fusion`, `distribution_based_rank_fusion`, `concatenate`, `merge`) instead of Weaviate's continuous `alpha`. RRF is rank-based and robust to score-scale differences between BM25 and dense — a good default for most use cases.